#PDF Loader

In [ ]:
%%capture
!pip install langchain langchain-community langchain-openai faiss-cpu pypdf

In [ ]:
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_KEY")


from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def create_policy_qa_chain(pdf_path: str):

    # Load PDF
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # Split into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)

    # Embeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # Vector Store
    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 3}
    )

    # Groq LLM
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0
    )

    prompt = ChatPromptTemplate.from_template(
        """
You are an assistant for question-answering tasks.

Use only the provided context.

If the answer is not available in the context,
say "I don't know".

Context:
{context}

Question:
{question}
"""
    )

    def format_docs(docs):
        return "\n\n".join(
            doc.page_content for doc in docs
        )

    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

In [ ]:
pdf_file = "/content/Deep_TimeSeries_EDA_Report.pdf"

qa_chain = create_policy_qa_chain(pdf_file)

question = "What is the Descriptive Statistics in document?"

response = qa_chain.invoke(question)

print(response)

print()
print()

question = "What is the document regarding?"

response = qa_chain.invoke(question)

print(response)


print()
print()

question = "What is the total strength?"

response = qa_chain.invoke(question)

print(response)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

The Descriptive Statistics are as follows:

- count: 7423.000000
- mean: 7.390210
- std: 88.544662
- min: -232.529984
- 25%: -54.934158
- 50%: -13.113405
- 75%: 73.771633
- max: 441.643674


The document appears to be a Deep Time Series Exploratory Data Analysis (EDA) Report, focusing on analyzing a time series dataset, specifically the "red_corrected" variable.


I don't know.
